In [1]:
from pathlib import Path
import sys


def find_repository_root() -> Path:
    
    candidate = Path.cwd().resolve()

    while candidate != candidate.parent:

        if ((candidate / "app").is_dir() and (candidate / "notebooks").is_dir()):
            
            return candidate

        candidate = candidate.parent

    raise RuntimeError("Could not locate reconciliation_engine repository root.")

PROJECT_ROOT = find_repository_root()

if str(PROJECT_ROOT) not in sys.path:
    
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Repository root: {PROJECT_ROOT}")

Repository root: C:\Users\ASUS\Desktop\Projects\reconciliation_engine


In [2]:
from pathlib import Path

import pandas as pd

from app.core.config import settings
from app.reporting.report_generator import ReportGenerator
from app.services.reconciliation_service import ReconciliationService
from app.services.fuzzy_match_service import fuzzy_match_field

In [3]:
class DemoSalesClient:

    def get_sales_records(self):

        return pd.DataFrame([
                {
                    "transaction_id": "101",
                    "invoice_number": "INV-2026-000101",
                    "product_id": "10",
                    "quantity": "5",
                    "status": "VALIDATED",
                    "created_at": "2026-08-08T09:00:00Z"
                },
                {
                    "transaction_id": "102",
                    "invoice_number": "INV-2026-000102",
                    "product_id": "11",
                    "quantity": "10",
                    "status": "COMPLETED",
                    "created_at": "2026-08-08T10:00:00Z"
                },
                {
                    "transaction_id": "103",
                    "invoice_number": "INV-2026-000103",
                    "product_id": "12",
                    "quantity": "5",
                    "status": "VALIDATED",
                    "created_at": "2026-08-08T11:00:00Z"
                },
                {
                    "transaction_id": "104",
                    "invoice_number": "INV-2026-000104",
                    "product_id": "13",
                    "quantity": "3",
                    "status": "VALIDATED",
                    "created_at": "2026-08-08T12:00:00Z"
                }])

In [4]:
class DemoInventoryClient:

    def get_inventory_records(self):

        return pd.DataFrame([
                # MATCHED
                {
                    "transaction_id": "101",
                    "reservation_id": "501",
                    "batch_id": "20",
                    "reserved_quantity": "5",
                    "status": "RESERVED",
                    "reserved_at": "2026-08-08T09:01:00Z"
                },
                # QUANTITY_MISMATCH
                {
                    "transaction_id": "102",
                    "reservation_id": "502",
                    "batch_id": "21",
                    "reserved_quantity": "7",
                    "status": "RESERVED",
                    "reserved_at": "2026-08-08T10:01:00Z"
                },
                # DUPLICATE_RESERVATION
                {
                    "transaction_id": "103",
                    "reservation_id": "503",
                    "batch_id": "22",
                    "reserved_quantity": "3",
                    "status": "RESERVED",
                    "reserved_at": "2026-08-08T11:01:00Z"
                },
                {
                    "transaction_id": "103",
                    "reservation_id": "504",
                    "batch_id": "23",
                    "reserved_quantity": "2",
                    "status": "RESERVED",
                    "reserved_at": "2026-08-08T11:02:00Z"
                },
                # ORPHAN_RESERVATION
                {
                    "transaction_id": "105",
                    "reservation_id": "505",
                    "batch_id": "24",
                    "reserved_quantity": "6",
                    "status": "RESERVED",
                    "reserved_at": "2026-08-08T13:01:00Z"
                }])

In [5]:
sales_client = DemoSalesClient()
inventory_client = DemoInventoryClient()

report_generator = ReportGenerator()

service = ReconciliationService(
    sales_client=sales_client,
    inventory_client=inventory_client,
    report_generator=report_generator
)

In [6]:
result = service.run()

result.keys()

dict_keys(['source_df', 'target_df', 'comparison_results', 'comparison_df', 'mismatches', 'fuzzy_matches', 'summary_df'])

In [7]:
sales_columns = [
    "transaction_id",
    "invoice_number",
    "product_id",
    "quantity",
    "status",
    "created_at"
]

assert list(result["source_df"].columns) == sales_columns

display(result["source_df"])

,transaction_id,invoice_number,product_id,quantity,status,created_at
0,101,INV-2026-000101,10,5,VALIDATED,2026-08-08 09:00:00+00:00
1,102,INV-2026-000102,11,10,COMPLETED,2026-08-08 10:00:00+00:00
2,103,INV-2026-000103,12,5,VALIDATED,2026-08-08 11:00:00+00:00
3,104,INV-2026-000104,13,3,VALIDATED,2026-08-08 12:00:00+00:00


In [8]:
inventory_columns = [
    "transaction_id",
    "reservation_id",
    "batch_id",
    "reserved_quantity",
    "status",
    "reserved_at",
]

assert list(result["target_df"].columns) == inventory_columns

display(result["target_df"])

,transaction_id,reservation_id,batch_id,reserved_quantity,status,reserved_at
0,101,501,20,5,RESERVED,2026-08-08 09:01:00+00:00
1,102,502,21,7,RESERVED,2026-08-08 10:01:00+00:00
2,103,503,22,3,RESERVED,2026-08-08 11:01:00+00:00
3,103,504,23,2,RESERVED,2026-08-08 11:02:00+00:00
4,105,505,24,6,RESERVED,2026-08-08 13:01:00+00:00


In [9]:
final_results = pd.DataFrame(result["comparison_results"])

display(
    final_results[
        [
            "transaction_id",
            "status",
            "mismatch_type"
        ]])

,transaction_id,status,mismatch_type
0,101,MATCHED,NaN
1,102,MISMATCHED,QUANTITY_MISMATCH
2,103,MISMATCHED,DUPLICATE_RESERVATION
3,104,MISMATCHED,MISSING_RESERVATION
4,105,MISMATCHED,ORPHAN_RESERVATION


In [10]:
results_by_transaction = {int(result["transaction_id"]): result for result in result["comparison_results"]}

assert (results_by_transaction[102]["mismatch_type"] == "QUANTITY_MISMATCH")

assert (results_by_transaction[103]["mismatch_type"] == "DUPLICATE_RESERVATION")

assert (results_by_transaction[104]["mismatch_type"] == "MISSING_RESERVATION")

assert (results_by_transaction[105]["mismatch_type"] == "ORPHAN_RESERVATION")

print("All required discrepancy classifications verified.")

All required discrepancy classifications verified.


In [11]:
duplicate_result = results_by_transaction[103]

assert duplicate_result["mismatch_type"] == ("DUPLICATE_RESERVATION")

assert duplicate_result["reservation_count"] == 2

assert duplicate_result["reservation_ids"] == ["503", "504"]

assert duplicate_result["reserved_quantities"] == [3, 2]

print("Duplicate reservation preserved:", duplicate_result["reservation_ids"])

Duplicate reservation preserved: ['503', '504']


In [12]:
assert isinstance(result["fuzzy_matches"], dict)

assert result["fuzzy_matches"] == {}

print("Fuzzy-matching stage executed.")

print("No fuzzy matches were produced because the current Sales/Inventory reconciliation contracts contain no supported shared textual field.")

Fuzzy-matching stage executed.
No fuzzy matches were produced because the current Sales/Inventory reconciliation contracts contain no supported shared textual field.


In [13]:
fuzzy_result = fuzzy_match_field("ABC Corporation", "ABC Corp", field_name="secondary_text", threshold=65)

display(pd.DataFrame([fuzzy_result]))

assert fuzzy_result["field"] == "secondary_text"
assert fuzzy_result["matched"] is True
assert fuzzy_result["threshold"] == 65
assert fuzzy_result["score"] >= 65

,field,left_value,right_value,score,threshold,matched
0,secondary_text,ABC Corporation,ABC Corp,69.565217,65,True


In [14]:
display(pd.DataFrame(result["mismatches"]))

,transaction_id,mismatch_type,invoice_number,sales_quantity,reserved_quantity,reservation_count,reservation_id,batch_id,reservation_ids,batch_ids,reserved_quantities,reservation_statuses,reservation_timestamps,details,reservations
0,104,MISSING_RESERVATION,INV-2026-000104,3.0,NaN,0,NaN,NaN,[],[],[],[],[],Sales transaction has no corresponding Invento...,NaN
1,103,DUPLICATE_RESERVATION,NaN,NaN,5.0,2,NaN,NaN,"[503, 504]","[22, 23]","[3, 2]","[RESERVED, RESERVED]","[2026-08-08 11:01:00+00:00, 2026-08-08 11:02:0...",Multiple Inventory reservations exist for the ...,"[{'reservation_id': '503', 'batch_id': '22', '..."
2,102,QUANTITY_MISMATCH,INV-2026-000102,10.0,7.0,1,502,21,[502],[21],[7],[RESERVED],[2026-08-08 10:01:00+00:00],Sales transaction quantity does not match Inve...,"[{'reservation_id': '502', 'batch_id': '21', '..."
3,105,ORPHAN_RESERVATION,NaN,NaN,6.0,1,NaN,NaN,[505],[24],[6],[RESERVED],[2026-08-08 13:01:00+00:00],Inventory reservation references a transaction...,"[{'reservation_id': '505', 'batch_id': '24', '..."


In [15]:
display(result["summary_df"])

,category_type,category,count
0,status,MISMATCHED,4
1,status,MATCHED,1
2,mismatch_type,QUANTITY_MISMATCH,1
3,mismatch_type,DUPLICATE_RESERVATION,1
4,mismatch_type,MISSING_RESERVATION,1
5,mismatch_type,ORPHAN_RESERVATION,1


In [16]:
REPORT_DIR = Path(settings.reports_dir).resolve()

CHART_DIR = Path(settings.charts_dir).resolve()

expected_reports = [
    REPORT_DIR / "matched.csv",
    REPORT_DIR / "mismatched.csv",
    REPORT_DIR / "missing.csv",
    REPORT_DIR / "summary.csv",
]

expected_chart = (CHART_DIR / "reconciliation_summary.png")

for path in expected_reports:
    
    print(path)

print(expected_chart)

C:\Users\ASUS\Desktop\Projects\reconciliation_engine\reports\matched.csv
C:\Users\ASUS\Desktop\Projects\reconciliation_engine\reports\mismatched.csv
C:\Users\ASUS\Desktop\Projects\reconciliation_engine\reports\missing.csv
C:\Users\ASUS\Desktop\Projects\reconciliation_engine\reports\summary.csv
C:\Users\ASUS\Desktop\Projects\reconciliation_engine\reports\charts\reconciliation_summary.png


In [17]:
for report_path in expected_reports:

    assert report_path.exists(), (f"Missing report: {report_path}")

assert expected_chart.exists(), (f"Missing chart: {expected_chart}")

print("All reconciliation reports and analytics chart exist.")

All reconciliation reports and analytics chart exist.


In [18]:
for report_path in expected_reports:

    print(f"\n--- {report_path.name} ---")

    display(pd.read_csv(report_path))


--- matched.csv ---


,transaction_id,invoice_number,status,mismatch_type,sales_status,inventory_status,quantity,reserved_quantity,reservation_count,reservation_id,batch_id,reservation_ids,batch_ids,reserved_quantities,reservation_statuses,reservation_timestamps,reservations
0,101,INV-2026-000101,MATCHED,NaN,VALIDATED,RESERVED,5,5,1,501,20,"[""501""]","[""20""]",[5],"[""RESERVED""]","[""2026-08-08 09:01:00+00:00""]","[{""reservation_id"": ""501"", ""batch_id"": ""20"", ""..."



--- mismatched.csv ---


,transaction_id,invoice_number,status,mismatch_type,sales_status,inventory_status,quantity,reserved_quantity,reservation_count,reservation_id,batch_id,reservation_ids,batch_ids,reserved_quantities,reservation_statuses,reservation_timestamps,reservations,sales_quantity,details
0,102,INV-2026-000102,MISMATCHED,QUANTITY_MISMATCH,COMPLETED,RESERVED,10.0,7,1,502.0,21.0,"[""502""]","[""21""]","[""7""]","[""RESERVED""]","[""2026-08-08 10:01:00+00:00""]","[{""reservation_id"": ""502"", ""batch_id"": ""21"", ""...",10.0,Sales transaction quantity does not match Inve...
1,103,NaN,MISMATCHED,DUPLICATE_RESERVATION,VALIDATED,"[""RESERVED"", ""RESERVED""]",5.0,5,2,NaN,NaN,"[""503"", ""504""]","[""22"", ""23""]","[3, 2]","[""RESERVED"", ""RESERVED""]","[""2026-08-08 11:01:00+00:00"", ""2026-08-08 11:0...","[{""reservation_id"": ""503"", ""batch_id"": ""22"", ""...",NaN,Multiple Inventory reservations exist for the ...
2,105,NaN,MISMATCHED,ORPHAN_RESERVATION,NaN,"[""RESERVED""]",NaN,6,1,NaN,NaN,"[""505""]","[""24""]",[6],"[""RESERVED""]","[""2026-08-08 13:01:00+00:00""]","[{""reservation_id"": ""505"", ""batch_id"": ""24"", ""...",NaN,Inventory reservation references a transaction...



--- missing.csv ---


,transaction_id,invoice_number,status,mismatch_type,sales_status,inventory_status,quantity,reserved_quantity,reservation_count,reservation_id,batch_id,reservation_ids,batch_ids,reserved_quantities,reservation_statuses,reservation_timestamps,sales_quantity,details
0,104,INV-2026-000104,MISMATCHED,MISSING_RESERVATION,VALIDATED,NaN,3,NaN,0,NaN,NaN,[],[],[],[],[],3,Sales transaction has no corresponding Invento...



--- summary.csv ---


,category_type,category,count
0,status,MISMATCHED,4
1,status,MATCHED,1
2,mismatch_type,QUANTITY_MISMATCH,1
3,mismatch_type,DUPLICATE_RESERVATION,1
4,mismatch_type,MISSING_RESERVATION,1
5,mismatch_type,ORPHAN_RESERVATION,1


In [19]:
mismatched_df = pd.read_csv(REPORT_DIR / "mismatched.csv")

missing_df = pd.read_csv(REPORT_DIR / "missing.csv")

assert set(mismatched_df["mismatch_type"]) == {
    "QUANTITY_MISMATCH",
    "DUPLICATE_RESERVATION",
    "ORPHAN_RESERVATION"
}

assert set(missing_df["mismatch_type"]) == {"MISSING_RESERVATION"}

print("Detailed mismatch classifications survived report serialization.")

Detailed mismatch classifications survived report serialization.


In [20]:
assert set(final_results["transaction_id"]) == {101, 102, 103, 104, 105}

assert (len(final_results) == 5)

assert (len(pd.read_csv(REPORT_DIR / "matched.csv")) == 1)

assert (len(pd.read_csv(REPORT_DIR / "mismatched.csv")) == 3)

assert (len(pd.read_csv(REPORT_DIR / "missing.csv")) == 1)

print("Reconciliation demonstration completed successfully.")
print(f"Reports: {REPORT_DIR}")
print(f"Chart: {expected_chart}")

Reconciliation demonstration completed successfully.
Reports: C:\Users\ASUS\Desktop\Projects\reconciliation_engine\reports
Chart: C:\Users\ASUS\Desktop\Projects\reconciliation_engine\reports\charts\reconciliation_summary.png
